# Step 1 — Data cleaning (Colab)

**এই নোটবুক যা করে**
1. সব রেকর্ডিং **16 kHz mono WAV** এ কনভার্ট করে, শুরু-শেষের লম্বা নীরবতা কেটে দেয়, আর `A2/A2_0002.wav` নিয়মে নাম দেয় (সংখ্যা = Excel row)
2. নষ্ট / খুব ছোট / খুব লম্বা ফাইল আলাদা করে রিপোর্ট দেয়
3. **নরমালাইজ করা টেক্সট** (`corpus.csv`) জুড়ে দেয়
4. **Train / validation / test** ভাগ করে — বাক্য অনুযায়ী (৮০/১০/১০), আর **A2 পুরোপুরি test এ** (unseen speaker)
5. `Step1b_Recording_Check` নোটবুক যদি ভুল-জায়গার রেকর্ডিং ঠিক করে থাকে (`recheck/corrections.csv`), সেটা প্রয়োগ করে


**শুরুর আগে:** `Ultimate Thisis Data` ফোল্ডারটা পুরো Google Drive এর **My Drive** এ থাকতে হবে (ভেতরে `Thesis Data 2`, `mapping.csv`, `corpus.csv`)। Drive এ আরও ~10 GB খালি জায়গা লাগবে। এই নোটবুকে GPU লাগবে না।

**চালানোর নিয়ম:** Runtime → Run all। মাঝপথে বন্ধ হলে আবার Run all — যেখানে থেমেছিল সেখান থেকে চলবে।

**ক্রম:** এই নোটবুক (সেল ১–৫) → `Step1b_Recording_Check.ipynb` → আবার এই নোটবুক Run all (তখন ঠিক করা নম্বরগুলো split এ ঢুকে যাবে)।

## 1. Google Drive যুক্ত করুন

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. সেটিংস

In [ ]:
ROOT = "/content/drive/MyDrive/Ultimate Thisis Data"   # folder with "Thesis Data 2", mapping.csv, corpus.csv
OUT  = "/content/wav_16k"                               # fast local Colab disk
SAVE = ROOT + "/wav_16k_tar"                            # finished speakers are kept here (survives disconnects)
SPEAKERS = ["A1", "A2", "A4", "A6", "A7"]   # speakers who withdrew consent were excluded
TEST_SPEAKER = "A2"   # unseen-speaker test set: never used for training

import os, pandas as pd
for f in ["Thesis Data 2", "mapping.csv", "corpus.csv"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"'{f}' not found in ROOT - upload it / check ROOT"
m = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
print("Files in mapping:", len(m)); print(m.status.value_counts().to_string())
print("\nRecordings to convert per speaker:")
print(m[m.status.isin(["primary", "second_voice"])].groupby("speaker").size().to_string())

## 3. কনভার্সন স্ক্রিপ্ট (কিছু বদলাতে হবে না)

In [ ]:
%%writefile /content/convert_to_wav.py
"""
Sylheti thesis dataset -> 16 kHz mono 16-bit WAV
==================================================

Reads mapping.csv (made from the audio-alignment audit) and converts every
recording to a clean, consistently named WAV file:

    wav_16k/<Speaker>/<Speaker>_<ExcelRow>.wav          (one take per sentence)
    wav_16k/<Speaker>/alt/<Speaker>_<ExcelRow>_altN.wav (extra takes, only with --include-alt)

It also writes, inside the output folder:
    metadata.csv           file_name, speaker, excel_row, sylheti, bangla, duration_sec, confidence
                           (Hugging Face "audiofolder" format: load_dataset("audiofolder", data_dir=...))
    conversion_report.csv  every file that failed, is too short (<0.4 s) or very long (>30 s)

Works on Windows, Mac, Linux and Google Colab. Needs Python 3.8+ and ffmpeg.
Safe to stop and re-run: files that already exist are skipped.

Usage (from the "Ultimate Thisis Data" folder):
    python convert_to_wav.py
    python convert_to_wav.py --trim-silence          # also cut silence at start/end
    python convert_to_wav.py --include-alt           # also convert extra takes
    python convert_to_wav.py --speakers A2 A7   # only some speakers
    python convert_to_wav.py --root "/content/drive/MyDrive/Ultimate Thisis Data"   # Colab
"""
import argparse, csv, os, shutil, subprocess, sys, time, wave
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

TRIM = ("silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.15,"
        "areverse,silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.15,areverse")


def find_ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    print("\n[!] ffmpeg was not found.\n"
          "    Windows: open PowerShell and run:  winget install Gyan.FFmpeg\n"
          "             then close and reopen PowerShell / the terminal.\n"
          "    Colab:   ffmpeg is already installed.\n"
          "    Mac:     brew install ffmpeg\n")
    sys.exit(1)


def wav_duration(path):
    try:
        with wave.open(str(path), "rb") as w:
            return w.getnframes() / float(w.getframerate())
    except Exception:
        return None


def convert(ffmpeg, src, dst, trim):
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.stem + ".part.wav")
    cmd = [ffmpeg, "-nostdin", "-hide_banner", "-loglevel", "error", "-y", "-i", str(src), "-vn"]
    if trim:
        cmd += ["-af", TRIM]
    cmd += ["-ac", "1", "-ar", "16000", "-sample_fmt", "s16", str(tmp)]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0 or not tmp.exists() or tmp.stat().st_size < 100:
        if tmp.exists():
            tmp.unlink()
        return (p.stderr or "ffmpeg failed").strip().splitlines()[-1:] or ["ffmpeg failed"]
    os.replace(tmp, dst)
    return None


def main():
    here = Path(__file__).resolve().parent
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--root", default=str(here), help='folder that contains "Thesis Data 2" (default: this script\'s folder)')
    ap.add_argument("--mapping", default=None, help="mapping.csv (default: <root>/mapping.csv)")
    ap.add_argument("--out", default=None, help="output folder (default: <root>/wav_16k)")
    ap.add_argument("--include-alt", action="store_true", help="also convert extra takes (status alt_take)")
    ap.add_argument("--trim-silence", action="store_true", help="cut silence at the start and end")
    ap.add_argument("--speakers", nargs="*", help="only these speakers")
    ap.add_argument("--workers", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    a = ap.parse_args()

    root = Path(a.root)
    mapping = Path(a.mapping) if a.mapping else root / "mapping.csv"
    out = Path(a.out) if a.out else root / "wav_16k"
    audio_root = root / "Thesis Data 2"
    ffmpeg = find_ffmpeg()
    if not mapping.exists():
        sys.exit(f"[!] mapping file not found: {mapping}")
    if not audio_root.exists():
        sys.exit(f'[!] folder not found: {audio_root}\n    Use --root to point at the folder that contains "Thesis Data 2".')

    wanted = {"primary", "second_voice"} | ({"alt_take"} if a.include_alt else set())
    jobs = []
    with open(mapping, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            if r["status"] not in wanted or not r["wav_name"]:
                continue
            if a.speakers and r["speaker"] not in a.speakers:
                continue
            src = audio_root.joinpath(*r["original_file"].replace("\\", "/").split("/"))
            dst = out.joinpath(*r["wav_name"].split("/"))
            jobs.append((r, src, dst))

    print(f"{len(jobs)} files to convert -> {out}   (workers: {a.workers}, trim silence: {a.trim_silence})")
    problems, done, t0 = [], 0, time.time()

    def work(job):
        r, src, dst = job
        if not src.exists():
            return job, "source file not found"
        if dst.exists() and wav_duration(dst):
            return job, None
        err = convert(ffmpeg, src, dst, a.trim_silence)
        return job, (" ".join(err) if err else None)

    with ThreadPoolExecutor(a.workers) as ex:
        for fut in as_completed([ex.submit(work, j) for j in jobs]):
            (r, src, dst), err = fut.result()
            done += 1
            if err:
                problems.append((r["speaker"], r["excel_row"], r["original_file"], "FAILED: " + err))
            if done % 500 == 0 or done == len(jobs):
                rate = done / max(time.time() - t0, 1e-6)
                print(f"  {done}/{len(jobs)}  ({rate:.0f} files/s, ~{(len(jobs)-done)/max(rate,1e-6)/60:.0f} min left)")

    # metadata + quality checks
    out.mkdir(parents=True, exist_ok=True)
    total = 0.0
    with open(out / "metadata.csv", "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file_name", "speaker", "excel_row", "status", "sylheti", "bangla", "duration_sec", "confidence"])
        for r, src, dst in jobs:
            d = wav_duration(dst) if dst.exists() else None
            if d is None:
                continue
            total += d
            if d < 0.4:
                problems.append((r["speaker"], r["excel_row"], r["original_file"], f"too short ({d:.2f}s)"))
            elif d > 30:
                problems.append((r["speaker"], r["excel_row"], r["original_file"], f"very long ({d:.1f}s)"))
            w.writerow([dst.relative_to(out).as_posix(), r["speaker"], r["excel_row"], r["status"],
                        r["sylheti"], r["bangla"], f"{d:.3f}", r["confidence"]])
    with open(out / "conversion_report.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(["speaker", "excel_row", "original_file", "problem"])
        w.writerows(sorted(problems))
    print(f"\nDone. Total audio: {total/3600:.1f} hours.  Problems: {len(problems)} "
          f"(see {out / 'conversion_report.csv'})")


if __name__ == "__main__":
    main()

## 4. সব অডিও WAV এ কনভার্ট
১–৩ ঘণ্টা লাগতে পারে। প্রতিটা speaker শেষ হলেই Drive এর `wav_16k_tar` এ সেভ হয়, তাই disconnect হলে আবার Run all দিলে শেষ হওয়া speaker আর কনভার্ট হবে না।

In [ ]:
import os, tarfile
def restore_wavs():
    """Bring the converted WAVs back from Drive after a Colab reset (takes a few minutes)."""
    os.makedirs(OUT, exist_ok=True)
    for s in SPEAKERS:
        t = f"{SAVE}/{s}.tar"
        if os.path.exists(t) and not os.path.isdir(f"{OUT}/{s}"):
            with tarfile.open(t) as f: f.extractall(OUT)
            print("restored", s)

!ffmpeg -version | head -1
os.makedirs(SAVE, exist_ok=True); restore_wavs()
for s in SPEAKERS:
    tar_path = f"{SAVE}/{s}.tar"
    if os.path.exists(tar_path):
        print(f"{s}: already done"); continue
    print(f"\n===== {s} =====")
    !python /content/convert_to_wav.py --root "{ROOT}" --out "{OUT}" --speakers {s} --trim-silence
    os.makedirs(f"{OUT}/{s}", exist_ok=True)
    with tarfile.open(tar_path + ".part", "w") as t:
        t.add(f"{OUT}/{s}", arcname=s)
    os.replace(tar_path + ".part", tar_path)
    print(f"{s}: saved to Drive")

## 5. metadata আর সমস্যার রিপোর্ট

In [ ]:
!python /content/convert_to_wav.py --root "{ROOT}" --out "{OUT}"
import shutil
for f in ["metadata.csv", "conversion_report.csv"]:
    shutil.copy(f"{OUT}/{f}", f"{SAVE}/{f}")
meta = pd.read_csv(OUT + "/metadata.csv")
s = meta.groupby("speaker").duration_sec.agg(files="count", hours="sum", mean_sec="mean", min_sec="min", max_sec="max")
s["hours"] /= 3600
print(s.round(2).to_string()); print("\nTotal:", len(meta), "files,", round(meta.duration_sec.sum() / 3600, 1), "hours")
rep = pd.read_csv(OUT + "/conversion_report.csv", encoding="utf-8-sig")
print(len(rep), "files with a problem (failed / too short / very long) — they are dropped below")
rep.head(20)

## 6. নরমালাইজ করা টেক্সট + ভুল রেকর্ডিং ঠিক করা + split
- টেক্সট আসে `corpus.csv` থেকে: Unicode (NFC), zero-width অক্ষর, কোটেশন/ড্যাশ, স্পেস ঠিক করা। `text` কলাম = যতিচিহ্ন ছাড়া সিলেটি (ASR এর লক্ষ্য)।
- বাক্যের split (`train/val/test`) `corpus.csv` এ আগে থেকেই ঠিক করা (seed 13), একই বাক্য বারবার থাকলে সব এক ভাগে। MT নোটবুকও একই split ব্যবহার করবে।
- `recheck/corrections.csv` থাকলে: যে রেকর্ডিং ভুল নম্বরে ছিল তার excel_row বদলে ঠিক বাক্য বসানো হয়। আর `recheck/drop.csv` এর ফাইল বাদ যায়।
- **test_seen** = অন্য speaker রা, test বাক্যে · **test_unseen** = A2, test বাক্যে · A2 এর বাকি রেকর্ডিং **unused**

In [ ]:
meta = pd.read_csv(OUT + "/metadata.csv")
corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
bad = set(pd.read_csv(OUT + "/conversion_report.csv", encoding="utf-8-sig").get("original_file", []))
fix_f, drop_f = ROOT + "/recheck/corrections.csv", ROOT + "/recheck/drop.csv"
if os.path.exists(fix_f):
    fx = pd.read_csv(fix_f, encoding="utf-8-sig").set_index("wav_name").suggested_row
    hit = meta.file_name.isin(fx.index)
    meta.loc[hit, "excel_row"] = meta.loc[hit, "file_name"].map(fx)
    print("corrections applied:", int(hit.sum()))
else:
    print("no recheck/corrections.csv yet (run Step1b_Recording_Check to find misplaced recordings)")
if os.path.exists(drop_f):
    d = set(pd.read_csv(drop_f, encoding="utf-8-sig").wav_name); meta = meta[~meta.file_name.isin(d)]
    print("left out (recheck/drop.csv):", len(d))
m = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
problem_wavs = set(m[m.original_file.isin(bad)].wav_name)
meta = meta[~meta.file_name.isin(problem_wavs)]
meta = meta.drop(columns=["sylheti", "bangla"]).merge(
    corpus[["excel_row", "split", "sylheti_asr", "sylheti", "bangla", "english"]], on="excel_row", how="left")
held = meta.speaker == TEST_SPEAKER
meta["split"] = meta.split.map({"train": "train", "val": "validation", "test": "test_seen"})
meta.loc[held, "split"] = meta.loc[held, "split"].map({"test_seen": "test_unseen"}).fillna("unused")
meta = meta.rename(columns={"sylheti_asr": "text"})
meta.to_csv(SAVE + "/metadata_splits.csv", index=False)
t = meta.groupby("split").agg(files=("file_name", "count"), hours=("duration_sec", lambda x: x.sum() / 3600),
                              speakers=("speaker", "nunique"), sentences=("excel_row", "nunique"))
print(t.round(2).to_string()); print("\nSaved:", SAVE + "/metadata_splits.csv")

কয়েকটা রেকর্ডিং শুনে মিলিয়ে দেখুন:

In [ ]:
from IPython.display import Audio, display
for _, r in meta.sample(5).iterrows():
    print(f"{r.speaker} row {r.excel_row} [{r.split}]  {r.sylheti}"); display(Audio(OUT + "/" + r.file_name))

## 7. ASR নোটবুকের জন্য `My Drive/sylheti_asr/metadata.csv`

In [ ]:
ASR_DIR = "/content/drive/MyDrive/sylheti_asr"; os.makedirs(ASR_DIR, exist_ok=True)
asr = meta[meta.split != "unused"].assign(audio_path=OUT + "/" + meta.file_name)
asr = asr[["audio_path", "text", "split", "speaker", "excel_row", "sylheti", "bangla", "english", "duration_sec"]]
asr.to_csv(ASR_DIR + "/metadata.csv", index=False)
print(len(asr), "rows ->", ASR_DIR + "/metadata.csv"); print(asr.split.value_counts().to_string())

---
**পরের ধাপ:** Step 2 (ASR)। নতুন Colab session এ WAV ফিরিয়ে আনতে সেল ১, ২, ৪ চালালেই হবে (Drive এর `.tar` থেকে কয়েক মিনিটে আসে)।